# Read the paired source-level evidence

The complete experiment is exploratory. Source bootstrap intervals condition on fixed fitted folds; they are not patient-level clinical confidence intervals. All declared comparisons remain visible.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/restart_data.py').exists()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/restart_data.py').exists())
sys.path.insert(0, str(PROJECT))
from worldmodels.settings import configure
PROJECT, REPOSITORY = configure()
DATA = PROJECT/'artifacts/runs/restart-full10hz-v1'
RUN = PROJECT/'artifacts/runs/restart-dynamics-v1'
assert (DATA/'preparation.json').exists(), 'Run the full extraction command in README first.'
meta = pd.read_csv(DATA/'windows.csv')
with np.load(DATA/'windows.npz', allow_pickle=False) as z:
    x, valid = z['x'], z['valid']
print('Real GAVD observations:', x.shape, 'sources:', meta.source_id.nunique())


Compare errors in the declared coordinate units. Lower error is better. Horizon eight averages the eight nominal future samples from 0.1 to 0.8 seconds; it is not an endpoint-only error. Three prepared windows fail the common distal-target support gate.

In [ ]:
results=json.loads((RUN/'results.json').read_text())
display(pd.DataFrame(results['normal_forecast']).query('horizon == 8'))

A favorable difference must be read alongside its interval and control. We separate forecast utility from evidence that correct bilateral timing matters.

In [ ]:
display(pd.DataFrame(results['paired_contrasts']).query('horizon == 8'))

All anomaly directions were fixed before evaluation. Higher scores mean more abnormal even if a score ranks the annotations poorly. Pooling out-of-fold rankings assumes score comparability across separately fitted normal-source models.

In [ ]:
display(pd.DataFrame(results['anomaly']).T)
print('Abnormality labels are annotations, not verified clinical diagnoses.')

Condition tables and source distributions reveal heterogeneity that a single pooled number conceals.

In [ ]:
condition=pd.read_csv(RUN/'condition_results.csv');display(condition.query('horizon == 8'))
source=pd.read_csv(RUN/'source_scores.csv')
fig,ax=plt.subplots(figsize=(9,4))
for label,group in source.query("arm == 'bilateral' and horizon == 8").groupby('label'):
    ax.scatter([label]*len(group),group.mse,alpha=.65)
ax.set(ylabel='Source-average articulation MSE');ax.tick_params(axis='x',rotation=20);plt.show()